# 17b — open-appsec: sessiya üzrə kənarda saxlanmış test

**Sual.** Notebook 17-də v3-ün open-appsec hissəsi bütün 692 sessiyanın dəyərlərindən təsadüfi götürülüb, ona görə
open-appsec test hissəsindəki aşağı FPR (0.01–0.13%) qismən "eyni saytı görmüş model" effekti ola bilər.
Burada sessiyaların bir hissəsi **tam kənarda** saxlanır və modellər onları heç görmədən yoxlanılır.

**Dizayn.**
1. 692 brauzer sessiyasından (open-appsec `legitimate.zip`, hər sessiya bir JSON faylı) **20%-i** (seed 42) kənarda saxlanır.
2. Kənar sessiyalarda görünən **hər dəyər** v3-ün open-appsec nümunə hovuzundan çıxarılır; bundan əlavə, hər hansı mənbədə sorğusu
   bu dəyərlərdən biri ilə eyni olan sətir də v3-dən silinir (`v3.clean(..., holdout=...)`). Beləliklə kənar sessiyaların heç bir dəyəri
   train/val-a düşmür (kod `assert` ilə yoxlayır).
3. Qalan hər şey 17 ilə eynidir: open-appsec-dən 40 000 unikal təmiz dəyər (seed 42), təmizləmə, mənbə üzrə qrup əsaslı 70/15/15,
   10 konfiqurasiya (LR / SVM / RF / NB / CNN × baza / norm.), seed 42–46, CNN 5 epoxa.
4. Kənar sessiyalarda FPR notebook 15-dəki kimi: unikal dəyər, çəkili dəyər (təkrarlarla), sorğu (≥1 dəyəri olan sorğular) və bütün sorğular.
   Eyni cədvəldə: **libinjection 4.0.1** və müqayisə üçün v2 modelləri (Kaggle-da öyrədilmiş LR baza və yerləşdirilən LR + norm. + aug.;
   proqnozlar notebook 15-in keşindən).

**Vaxt.** Hər seed ≈ 1 saat (CNN təlimi ≈ 42 dəq); RF və NB TF-IDF vektorlaşdırıcısını LR/SVM ilə paylaşdığından onları çıxarmaq vaxtı demək olar
dəyişmir, ona görə 10 konfiqurasiyanın hamısı saxlanılıb. Modellər `models/v3/*_sess_*`, proqnozlar `models/v3/sess_preds/`-də keşlənir.

In [ ]:
import sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import v3
import openappsec as oa

RES, FIG = v3.RES, v3.FIG
for d in (RES, FIG, v3.MODELS):
    d.mkdir(parents=True, exist_ok=True)
SEEDS = [42, 43, 44, 45, 46]
SESSION_SEED = 42
HOLDOUT_FRAC = 0.20
CNN_EPOCHS = 5
TAG = "sess"
PRED_CACHE = v3.MODELS / "sess_preds"
PRED_CACHE.mkdir(parents=True, exist_ok=True)
V2_CACHE = ROOT / "models" / "15_preds"                 # notebook 15-in proqnozları (v2 modelləri, libinjection)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#8a8984",
    "axes.labelcolor": "#2b2b2a", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.titleweight": "bold", "axes.titlesize": 10, "legend.frameon": False,
    "grid.color": "#e4e3df", "grid.linewidth": 0.6,
})

## 1. Kənar sessiyalar

In [ ]:
requests, values = oa.load()
sessions = np.array(sorted(requests.site.unique()))
n_hold = int(round(HOLDOUT_FRAC * len(sessions)))
held_sessions = set(np.random.RandomState(SESSION_SEED).choice(sessions, n_hold, replace=False))
pd.Series(sorted(held_sessions), name="session").to_csv(RES / "17b_holdout_sessions.csv", index=False)

# 15-dəki kimi: bütün dəyərlər üzrə qlobal unikal kodlar (15-in proqnoz keşi bu sıra ilə saxlanılıb)
codes, uniq = pd.factorize(values.value)
held_req = requests.site.isin(held_sessions).to_numpy()
val_held = held_req[values.req_id.to_numpy()]
held_set = set(values.value[val_held])

h_codes = codes[val_held]                                # kənar sessiyalardakı dəyər nüsxələri (qlobal kod)
h_ids = np.unique(h_codes)                               # kənar sessiyaların unikal dəyərləri (qlobal kod)
h_local = np.searchsorted(h_ids, h_codes)                # nüsxə -> lokal unikal indeks
h_req_ids = values.req_id.to_numpy()[val_held]
h_requests = requests[held_req]
req_pos = pd.Series(np.arange(len(h_requests)), index=h_requests.req_id.to_numpy())
h_req_local = req_pos.loc[h_req_ids].to_numpy()          # nüsxə -> kənar sorğu indeksi
h_has_val = h_requests.n_values.to_numpy() > 0
H = pd.Series(uniq[h_ids])
h_occ = np.bincount(h_local, minlength=len(h_ids))

summary = pd.Series({
    "sessiyalar (cəmi)": len(sessions), "kənar sessiyalar": n_hold,
    "kənar sorğular": int(held_req.sum()), "kənar sorğular (≥1 dəyər)": int(h_has_val.sum()),
    "kənar dəyər nüsxələri": int(val_held.sum()), "kənar unikal dəyərlər": len(h_ids),
    "kənar unikal dəyərlərin qalan sessiyalarda da görünən payı, %":
        round(100 * np.isin(h_ids, np.unique(codes[~val_held])).mean(), 2),
})
summary.to_csv(RES / "17b_holdout_summary.csv", header=["value"])
summary.to_frame("dəyər")

Qeyd: kənar sessiyaların unikal dəyərlərinin bir hissəsi (məs. `0`, `1`, `true`) qalan sessiyalarda da görünür. Onlar da v3-ün
open-appsec hovuzundan çıxarılır, yəni model bu dəyərləri **heç bir open-appsec sessiyasından** öyrənmir.

## 2. Kənar sessiyalarsız v3 dataseti

In [ ]:
raw = v3.raw_sources(oa_exclude=held_set)
df, report = v3.clean(raw, holdout=held_set)
report.rename(index=v3.SOURCE_LABEL).to_csv(RES / "17b_cleaning_report.csv")
assert not df.Query.isin(held_set).any()
counts = df.groupby(["source", "Label"]).size().unstack(fill_value=0).reindex(v3.SOURCES).rename(index=v3.SOURCE_LABEL)
display(report.rename(index=v3.SOURCE_LABEL))
counts

## 3. Təlim (5 seed) və kənar sessiyalarda FPR

In [ ]:
def slug(name):
    return "".join(ch if ch.isalnum() else "_" for ch in name)

def predict_cached(model, name, seed, xs, n=50_000):
    f = PRED_CACHE / f"{slug(name)}_{seed}.npy"
    if f.exists():
        p = np.load(f)
        if len(p) == len(xs):
            return p
    p = np.concatenate([model.predict_score(xs[i:i + n])[0] for i in range(0, len(xs), n)]).astype(np.int8)
    np.save(f, p)
    return p

def fpr_rows(name, seed, pred_u, family):
    pred_occ = pred_u[h_local]
    blocked = np.bincount(h_req_local, weights=pred_occ, minlength=len(h_requests)) > 0
    return [{"seed": seed, "detector": name, "qrup": family, "metric": m, "value": v} for m, v in (
        ("fpr_value_unique", pred_u.mean()), ("fpr_value_weighted", pred_occ.mean()),
        ("fpr_request", blocked[h_has_val].mean()), ("fpr_request_all", blocked.mean()),
        ("blocked_requests", blocked.sum()))]

# İstinad detektorları: libinjection (deterministik) və v2 modelləri — notebook 15-in keşindən, kənar dəyərlərə məhdudlaşdırılır
REFS = {"libinjection 4.0.1": "libinjection_4_0_1_det", "v2: LR baza": "LR_baza_{s}",
        "v2: LR + norm. + aug. (yerləşdirilən)": "LR___norm____augmentation_{s}"}
rows, preds42 = [], {}
for name, key in REFS.items():
    for s in SEEDS:
        p = np.load(V2_CACHE / f"{key.format(s=s)}.npy")[h_ids]
        rows += fpr_rows(name, s, p, "istinad")
        if s == 42:
            preds42[name] = p

xs = H.tolist()
main_rows, model_info = [], []
for s in SEEDS:
    t0 = time.perf_counter()
    tr, va, te = v3.split(df, s)
    assert not (tr.Query.isin(held_set).any() or va.Query.isin(held_set).any())
    models, info = v3.train_models(s, tr, va, TAG, cnn=True, cnn_epochs=CNN_EPOCHS)
    model_info += [{"seed": s, "model": k, **v} for k, v in info.items()]
    slices = {"Birləşdirilmiş test": np.ones(len(te), bool), "Test: open-appsec (görülmüş sessiyalar)": (te.source == "openappsec").to_numpy()}
    main_rows += v3.cached_rows(f"sess_main_{s}", lambda: v3.evaluate_sliced(models, te, slices, s, "sess"))
    for name in v3.MODEL_ORDER:
        p = predict_cached(models[name], name, s, xs)
        rows += fpr_rows(f"v3-sess: {name}", s, p, "v3 (kənar sessiyalarsız)")
        if s == 42:
            preds42[f"v3-sess: {name}"] = p
    pd.DataFrame(rows).to_csv(RES / "17b_holdout_fpr_raw.csv", index=False)
    print(f"seed {s}: {time.perf_counter() - t0:.0f} s", flush=True)
raw_fpr = pd.DataFrame(rows)
main = pd.DataFrame(main_rows)
main.to_csv(RES / "17b_raw_main.csv", index=False)
pd.DataFrame(model_info).to_csv(RES / "17b_model_selection.csv", index=False)

In [ ]:
ORDER = [f"v3-sess: {m}" for m in v3.MODEL_ORDER] + list(REFS)
pct = raw_fpr[raw_fpr.metric != "blocked_requests"]
a = pct.groupby(["detector", "metric"]).value.agg(["mean", "std"]) * 100
a.round(4).to_csv(RES / "17b_holdout_fpr_mean_std.csv")
pm = lambda d: d["mean"].map("{:.3f}".format) + " ± " + d["std"].map("{:.3f}".format)
table = pm(a).unstack("metric").loc[ORDER, ["fpr_value_unique", "fpr_value_weighted", "fpr_request", "fpr_request_all"]]
table["bloklanan sorğu (orta)"] = (raw_fpr[raw_fpr.metric == "blocked_requests"].groupby("detector").value.mean()
                                   .loc[ORDER].round(0).astype(int))
table.to_csv(RES / "17b_holdout_fpr_table.csv")
print("Kənar sessiyalarda FPR, % (orta ± std, 5 seed; libinjection deterministikdir)")
table

In [ ]:
# Yoxlama: kənar sessiyalarsız öyrədilmiş modellər v3 testində (17 ilə müqayisə üçün)
METRICS = ["accuracy", "precision", "recall", "f1", "roc_auc", "fpr"]
g = main.groupby(["test", "input", "model"])[METRICS].agg(["mean", "std"])
g.to_csv(RES / "17b_test_mean_std.csv")
def f(t, inp, m):
    x = g.loc[(t, inp)]
    return [f"{100 * x.loc[k, (m, 'mean')]:.2f} ± {100 * x.loc[k, (m, 'std')]:.2f}" for k in v3.MODEL_ORDER]
chk = pd.DataFrame({"F1 (birləşdirilmiş)": f("Birləşdirilmiş test", "normal", "f1"),
                    "FPR (birləşdirilmiş)": f("Birləşdirilmiş test", "normal", "fpr"),
                    "Recall, alt. ws. (birləşdirilmiş)": f("Birləşdirilmiş test", "alt_whitespace", "recall"),
                    "FPR (open-appsec test, görülmüş sessiyalar)": f("Test: open-appsec (görülmüş sessiyalar)", "normal", "fpr")},
                   index=v3.MODEL_ORDER)
chk.to_csv(RES / "17b_test_table.csv")
chk

## 4. Top-20 yanlış blok, səbəb qrupları, sessiyalar üzrə (seed 42)

In [ ]:
h_req_count = (pd.DataFrame({"r": h_req_local, "c": h_local}).drop_duplicates().c
               .value_counts().reindex(range(len(h_ids)), fill_value=0).to_numpy())
req_mean = a.xs("fpr_request", level="metric")["mean"]
v3_req = req_mean.loc[[f"v3-sess: {m}" for m in v3.MODEL_ORDER]]
TOP_MODELS = list(dict.fromkeys(["v3-sess: LR norm.", v3_req.idxmin()]))
print("Top-20 üçün modellər:", TOP_MODELS, "(LR norm. — yerləşdirilən modelin analoqu; ikincisi — ən aşağı sorğu FPR-li v3 modeli)")

def top_fp(name, k=20):
    fp = np.flatnonzero(preds42[name] == 1)
    fp = fp[np.argsort(-h_req_count[fp], kind="stable")][:k]
    t = pd.DataFrame({"value": H.iloc[fp].str[:120].values, "bloklanan_sorğu": h_req_count[fp], "tezlik": h_occ[fp],
                      "səbəb": H.iloc[fp].map(oa.reason).values})
    t["digər detektorlar da bloklayır"] = [", ".join(n for n in ORDER if n != name and preds42[n][i]) for i in fp]
    return t

pd.set_option("display.max_colwidth", 120)
tops = {}
for name in TOP_MODELS:
    tops[name] = top_fp(name)
    tops[name].to_csv(RES / f"17b_top20_false_blocks_{slug(name.replace('v3-sess: ', ''))}_seed42.csv", index=False)
    print(name); display(tops[name])

In [ ]:
rs = H.map(oa.reason).to_numpy()
rows = []
for name in ORDER:
    fp = preds42[name] == 1
    s = pd.Series(h_req_count[fp]).groupby(rs[fp]).sum()
    for r, v in s.items():
        rows.append({"detector": name, "səbəb": r, "unikal_dəyər": int((rs[fp] == r).sum()), "bloklanan_sorğu": int(v)})
reasons = pd.DataFrame(rows)
reasons.to_csv(RES / "17b_fp_reasons_seed42.csv", index=False)

site = h_requests.site.to_numpy()
rows = []
for name in ORDER:
    blocked = np.bincount(h_req_local, weights=preds42[name][h_local], minlength=len(h_requests)) > 0
    s = pd.Series(blocked[h_has_val]).groupby(site[h_has_val]).mean()
    rows.append({"detector": name, "sessiyalar": len(s), "FPR>0 olan sessiyalar": int((s > 0).sum()),
                 "FPR>1% olan sessiyalar": int((s > 0.01).sum()), "max sessiya FPR %": round(100 * s.max(), 2),
                 "median sessiya FPR %": round(100 * s.median(), 3)})
by_sess = pd.DataFrame(rows).set_index("detector")
by_sess.to_csv(RES / "17b_fpr_by_session_seed42.csv")
display(reasons.pivot_table(index="səbəb", columns="detector", values="bloklanan_sorğu", fill_value=0)[ORDER])
by_sess

In [ ]:
labels = [n.replace("v3-sess: ", "v3 ").replace("v2: ", "v2 ").replace(" (yerləşdirilən)", "\n(yerləşd.)")
          .replace("libinjection 4.0.1", "libinjection").replace(" + norm. + aug.", "\n+norm.+aug.") for n in ORDER]
fig, ax = plt.subplots(figsize=(12, 4.2))
x, w = np.arange(len(ORDER)), 0.38
for i, (m, lab, col) in enumerate([("fpr_value_unique", "Unikal dəyər", "#2a78d6"),
                                    ("fpr_request", "Sorğu (≥1 dəyəri olan)", "#eb6834")]):
    d = a.xs(m, level="metric").loc[ORDER]
    b = ax.bar(x + (i - 0.5) * w, d["mean"], w - 0.03, yerr=d["std"].fillna(0), capsize=2, color=col, label=lab,
               error_kw=dict(elinewidth=0.7, ecolor="#52514e"))
    ax.bar_label(b, fmt="%.2f", fontsize=6, padding=2, color="#2b2b2a")
ax.axvline(len(v3.MODEL_ORDER) - 0.5, color="#c3c2b7", lw=0.8)
ax.set_yscale("symlog", linthresh=0.1)
ax.set_xticks(x, labels, fontsize=7, rotation=0)
ax.set_ylabel("FPR, % (orta ± std, 5 seed; simlog)")
ax.yaxis.grid(True); ax.set_axisbelow(True)
ax.set_title(f"Kənarda saxlanmış {n_hold} open-appsec sessiyasında yanlış bloklama", loc="left")
ax.legend(ncol=2, loc="upper left")
fig.savefig(FIG / "17b_holdout_fpr.png")
plt.show()